In [68]:
import warnings
warnings.filterwarnings("ignore")

In [69]:
import os
import torch
import numpy as np
import pandas as pd

from flonacomldft.utils.io_utils import (
    load_pickle_file,
    get_project_path
)

from flonacomldft.internal_coordinates import (
    Coordinates_mapping
)

from flonacomldft.utils.diagnostics import R_hat

# 1. Ab-flowMC simulations

In [70]:
simulation_types = ['ab-flowMC', 'ab-flowMC w.o. MLP']

In [71]:
simulation_idxs = { simulation_types[0]: {
                        0: 31959916,
                        1: 31959919
                    },

                    simulation_types[1]: {
                        0: 31961552,
                        1: 31961553
                    }
                }

In [72]:
simulations_folders = {
    simulation_types[0]: get_project_path() + "/2-adaptive-mlp",
    simulation_types[1]: get_project_path() + "/1-adaptive"
}

save_path = get_project_path() + "/figures/preprint_v2/data"

In [73]:
get_simulation_filename = lambda isomer_label, idx: "results_adaptive_is{:d}_{:d}/adaptive_sampling_is{:d}_{:d}.pkl".format(
    isomer_label, idx, isomer_label, idx)
get_cvs_filename = lambda isomer_label, idx: "results_adaptive_is{:d}_{:d}/cvs_is{:d}_{:d}.pkl".format(
    isomer_label, idx, isomer_label, idx)

In [74]:
adaptives = {}
cvss = {}

print("Loading data...")
for simulation in simulation_types:
    adaptives[simulation] = {}
    cvss[simulation] = {}
    for key, idx in simulation_idxs[simulation].items():
        adaptives[simulation][key] = load_pickle_file(get_simulation_filename(key, idx), 
        simulations_folders[simulation])
        cvss[simulation][key] = load_pickle_file(get_cvs_filename(key, idx),
        simulations_folders[simulation])
        print("Loaded adaptive sampling {:s} and cvs for isomer {:d} and idx {:d}".format(
            simulation, key, idx))

Loading data...


Loaded adaptive sampling ab-flowMC and cvs for isomer 0 and idx 31959916
Loaded adaptive sampling ab-flowMC and cvs for isomer 1 and idx 31959919
Loaded adaptive sampling ab-flowMC w.o. MLP and cvs for isomer 0 and idx 31961552
Loaded adaptive sampling ab-flowMC w.o. MLP and cvs for isomer 1 and idx 31961553


In [75]:
adaptive = adaptives[simulation_types[0]][0]

## 1.a Acceptance rate

In [76]:
accs_df = []

for simulation in simulation_types:
    
    for key, idx in simulation_idxs[simulation].items():

        df = pd.DataFrame(columns=['Method', 'Isomer', 'Step', 'Acceptance Rate'])
        accs = torch.cat(adaptives[simulation][key]['accs']).float().mean(dim=1).numpy()

        df['Acceptance Rate'] = accs
        df['Isomer'] = key
        df['Step'] = np.arange(0, len(accs))
        df['Method'] = simulation

        accs_df.append(df)

accs_df = pd.concat(accs_df)
accs_df.to_csv(save_path + "/acceptance_rates.csv", index=False)
print("Acceptance rates saved to {:s}".format(save_path + "/acceptance_rates.csv"))

Acceptance rates saved to /mnt/home/amolina/ceph/adaptive-flow-mc/figures/preprint_v2/data/acceptance_rates.csv


## 1.a Collective variables and Potential energy

In [77]:
cvs_us_df = []

for simulation in simulation_types:
    for isomer_label in [0, 1]:

        us = torch.cat(adaptives[simulation][isomer_label]['us']).reshape(1000, 50, 1)
        cvs = cvss[simulation][isomer_label]

        cvs_us = torch.cat([us, cvs], dim=2).numpy().reshape(50000, 3)

        df = pd.DataFrame(cvs_us, columns=['C', 'R', 'U'])
        df['Isomer'] = isomer_label
        df['Method'] = simulation

        cvs_us_df.append(df)

cvs_us_df = pd.concat(cvs_us_df)
cvs_us_df.to_csv(save_path + "/cvs_potential_energy.csv", index=False)
print("CVs and potential energy saved to {:s}".format(save_path + "/cvs_potential_energy.csv"))
cvs_us_df

CVs and potential energy saved to /mnt/home/amolina/ceph/adaptive-flow-mc/figures/preprint_v2/data/cvs_potential_energy.csv


,C,R,U,Isomer,Method
0,-6.773585,10.105214,2.457276,0,ab-flowMC
1,-6.750111,10.205934,2.442652,0,ab-flowMC
2,-6.764058,9.652035,2.473082,0,ab-flowMC
3,-6.732985,10.172332,2.427748,0,ab-flowMC
4,-6.748763,9.293897,2.517407,0,ab-flowMC
...,...,...,...,...,...
49995,-6.704685,10.828248,2.220704,1,ab-flowMC w.o. MLP
49996,-6.746437,10.425041,2.232781,1,ab-flowMC w.o. MLP
49997,-6.714196,11.065727,2.218863,1,ab-flowMC w.o. MLP
49998,-6.743561,11.145951,2.213711,1,ab-flowMC w.o. MLP


# Rhat

In [78]:
#isomer_label = 0
#simulation = simulation_types[0]

rhat_df = []

for simulation in simulation_types:
    for isomer_label in [0, 1]:

        us = torch.cat(adaptives[simulation][isomer_label]['us']).reshape(1000, 50, 1)
        cvs = cvss[simulation][isomer_label]
        
        time_slices = adaptives[simulation][isomer_label]['time_mcmc']
        time_flatten = [t for time_slice in time_slices for t in time_slice]
        time_min = min(time_flatten)
        time_mcmc = np.array([t - time_min for t in time_flatten])/3600

        rhat_info = torch.cat((cvs, us), dim=2).detach().numpy().transpose(1, 0, 2)

        rhat = R_hat(rhat_info, labels=['C', 'R', 'U'])

        df = pd.DataFrame(rhat[0], columns=['C', 'R', 'U'])
        df['Time (h)'] = time_mcmc[rhat[1]]
        df['MCMC step'] = rhat[1]
        df['Method'] = simulation
        df['Isomer'] = isomer_label

        rhat_df.append(df)

rhat_df = pd.concat(rhat_df)
rhat_df.to_csv(save_path + "/rhat.csv", index=False)        
rhat_df

,C,R,U,Time (h),MCMC step,Method,Isomer
0,1.210686,1.228267,1.194109,2.800454,90,ab-flowMC,0
1,1.121994,1.119943,1.107089,5.679586,180,ab-flowMC,0
2,1.065124,1.070826,1.065830,8.531614,270,ab-flowMC,0
3,1.042406,1.042423,1.055567,11.332435,360,ab-flowMC,0
4,1.029414,1.032330,1.040203,14.137912,450,ab-flowMC,0
5,1.028392,1.029847,1.033139,16.935509,540,ab-flowMC,0
6,1.021719,1.023650,1.026424,19.749465,630,ab-flowMC,0
7,1.018697,1.021742,1.021554,22.615388,720,ab-flowMC,0
8,1.017255,1.021114,1.017018,25.403506,810,ab-flowMC,0
9,1.016924,1.018331,1.016177,28.190027,900,ab-flowMC,0


In [79]:
#